In [ ]:
import figure_settings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from figure_settings import style_settings

from conus_biomass import dir_info
from conus_biomass.make_figures import maps

# Load data

In [ ]:
# Load saved figure data
save_dir = "figure_data/figure_2/"

df_biomass = pd.read_csv(save_dir + "biomass_timeseries.csv")
biomass_mean_west = df_biomass.set_index("year")["biomass_mean"]
biomass_min_west = df_biomass.set_index("year")["biomass_min"]
biomass_max_west = df_biomass.set_index("year")["biomass_max"]

with open(save_dir + "target_crs.txt") as f:
    crs = f.read()

delta_biomass_clipped = xr.open_dataset(save_dir + "delta_biomass_clipped.nc")[
    "delta_biomass_clipped"
].rio.write_crs(crs)

western_states = maps.SHP_WESTERN.to_crs(crs)

# Make figure

In [ ]:
fig = plt.figure(figsize=(24, 10))

figure_settings.apply_style()

gs = fig.add_gridspec(1, 2, width_ratios=[1.3, 1], wspace=0.05)
axes = [fig.add_subplot(gs[0]), fig.add_subplot(gs[1])]

##############################################################################################################
##############################################################################################################
ax0 = axes[0]
ax0.plot(
    biomass_mean_west.index,
    (np.array(biomass_mean_west)),
    ".-",
    linewidth=5,
    color="#85a2f7",
    label="This Study",
)
ax0.fill_between(
    biomass_min_west.index,
    (np.array(biomass_min_west)),
    (np.array(biomass_max_west)),
    color="#85a2f7",
    alpha=0.4,
    edgecolor=None,
)

ax0.set_ylabel(
    "Live Aboveground Biomass (TgC)", labelpad=10, fontsize=style_settings["axes.titlesize"]
)
ax0.set_xlabel("Year", labelpad=10, fontsize=style_settings["axes.titlesize"])
ax0.spines[["right", "top"]].set_visible(True)
ax0.set_xlim([2004.5, 2022.5])
ax0.tick_params(axis="y", which="both", left=True, right=True, direction="in")
ax0.tick_params(axis="x", which="both", bottom=True, top=True, direction="in")

##############################################################################################################
##############################################################################################################

ax1 = axes[1]
maps.plot_map(
    delta_biomass_clipped,
    shp=western_states,
    latlon=False,
    title=None,
    cbar_label="Δ Live aboveground biomass (MgC/ha)",
    cmap=plt.cm.BrBG,
    clims=[-60, 60],
    savefig=None,
    ax=ax1,
)


for i, ax in enumerate(fig.axes):
    if ax != ax0 and ax != ax1:  # it's the colorbar axes
        ax.yaxis.label.set_fontsize(style_settings["axes.titlesize"])
        ax.tick_params(labelsize=style_settings["xtick.labelsize"])
        ax.set_ylabel("Δ Live aboveground biomass (MgC/ha) \n from 2005 to 2022", labelpad=10)

##############################################################################################################
##############################################################################################################


panel_labels = ["A", "B"]
for i, ax in enumerate(axes):
    ax.text(
        0.01,
        0.99,
        panel_labels[i],
        transform=ax.transAxes,
        fontsize=style_settings["font.size"] * 1.5,
        fontweight="bold",
        va="top",
    )

fig.subplots_adjust(left=0.05, right=0.92, top=0.92, bottom=0.08, wspace=-0.1)
fig.savefig(dir_info.dir_figures + "Figure2.jpg", bbox_inches="tight", dpi=300)